### Olist Silver ELT Scripts - NB 2

In [ ]:
# This notebook performs ELT/ETL operations on fact related tables in the
# olist datasets. The data is ingested from the bronze layer tables, cleaned, deduped and
# loaded to tables in the silver layer ready for data modelling and analysis

# Developer:    Asif Shah
# Created Date: 17.09.2026

In [ ]:
# import all required python libraries

from pyspark.sql import functions as f
from delta.tables import DeltaTable as dt
from pyspark.sql.types import IntegerType as int, DecimalType as dec

# Enable schema evolution for silver fact data:

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")


In [ ]:
# Order Review table cleanup and load into silver from bronze

# 1. read bronze table(s)
df_olist_ord_review = spark.read.format("delta").table("Bronze.olist_order_reviews")

# 2. deduplicate, correct data types and clean data
df_ordrev_cleanup = (df_olist_ord_review
    .drop_duplicates(["order_id", "review_id"])
    .withColumn("review_score", f.col("review_score").cast(int()))
    .fillna({"review_comment_title": "N/A", "review_comment_message": "N/A"})
    .withColumn("review_creation_date", f.col("review_creation_date").cast("timestamp"))
    .withColumn("review_answer_timestamp", f.col("review_answer_timestamp").cast("timestamp"))
    .withColumn("silver_ingested_at", f.current_timestamp())
    .withColumn("silver_updated_at", f.current_timestamp())
    .drop("ingestion_time")
)

# 3. view the final result - test before loading then comment out
# display(df_ordrev_cleanup)

# 4. Create table if not already exists
if not spark.catalog.tableExists("Silver.olist_order_reviews"):
    df_ordrev_cleanup.write.format("delta").saveAsTable("Silver.olist_order_reviews")

In [ ]:
# olist order review table merge

silver_ordrev_tbl = dt.forName(spark, "Silver.olist_order_reviews")

silver_ordrev_tbl.alias("target") \
  .merge (
    source = df_ordrev_cleanup.alias("source"),
    condition = "target.order_id = source.order_id and target.review_id = source.review_id"
                
          ) \
  .whenMatchedUpdate(
   set = {
        "review_score" : "source.review_score",
        "review_comment_title" : "source.review_comment_title",
        "review_comment_message" : "source.review_comment_message",
        "review_answer_timestamp" : "source.review_answer_timestamp",   
        "silver_updated_at" : f.current_timestamp()
   }
  ) \
  .whenNotMatchedInsertAll() \
  .execute()

In [ ]:
# payment table cleanup

# 1. read bronze table(s)
df_olist_bronze_payments = spark.read.format("delta").table("Bronze.olist_order_payments")
df_olist_silver_payments = spark.read.format("delta").table("Silver.olist_order_payments")

# 2. deduplicate, correct data types and clean data
df_pay_cleanup = (df_olist_bronze_payments
    .drop_duplicates(subset=["order_id", "payment_sequential"]) 
    .withColumn("payment_sequential", f.col("payment_sequential").cast(int()))
    .withColumn("payment_installments", f.col("payment_installments").cast(int()))
    .withColumn("payment_value", f.col("payment_value").cast("double"))
    .fillna({"payment_value": 0.00})
    .withColumn("payment_type", f.initcap(f.regexp_replace(f.col("payment_type"),"_"," ")))
    .withColumn("payment_type", f.initcap(f.col("payment_type")))
    .withColumn("silver_ingested_at", f.current_timestamp())
    .drop("ingestion_time")
    .join(df_olist_silver_payments, on=["order_id", "payment_sequential"], how="left_anti")
)

# 3. view the final result - test before loading then comment out
# display(df_pay_cleanup)

# 4. load data by appending to existing table or create new table via spark.sql or pyspark 
#    createdataframe method if required
df_pay_cleanup.write.format("delta").mode("append").saveAsTable("Silver.olist_order_payments")

In [ ]:
# order item table cleanup

# 1. read tables

df_bronze_olist_order_items = spark.read.format("delta").table("Bronze.olist_order_items")

# 2. deduplicate, correct data types and clean data
df_olist_order_clean = (df_bronze_olist_order_items
    .drop_duplicates(["order_id", "order_item_id"])
    .withColumn("order_item_id", f.col("order_item_id").cast(int()))
    .withColumn("shipping_limit_date", f.col("shipping_limit_date").cast("timestamp"))
    .withColumn("price", f.col("price").cast("double"))
    .withColumn("freight_value", f.col("freight_value").cast("double"))
    .fillna({"price": 0, "freight_value": 0})
    .withColumn("silver_ingested_at", f.current_timestamp())
    .withColumn("silver_updated_at", f.current_timestamp())
    .drop("ingestion_time")
  
)
# 3. view the final result - test before loading then comment out
# display(df_olist_order_clean)

# 4. load data by appending to existing table or create new table via spark.sql or pyspark 
#    createdataframe method if required
# df_olist_order_clean.write.format("delta").mode("append").saveAsTable("Silver.olist_order_items")




In [ ]:
# olist order items table merge

silver_orditems_tbl = dt.forName(spark, "Silver.olist_order_items")

silver_orditems_tbl.alias("target") \
  .merge (
    source = df_olist_order_clean.alias("source"),
    condition = "target.order_id = source.order_id and target.order_item_id = source.order_item_id"
                
          ) \
  .whenMatchedUpdate(
    condition="target.shipping_limit_date <> source.shipping_limit_date",
        set={"shipping_limit_date": "source.shipping_limit_date",
              "silver_updated_at": f.current_timestamp()}
   
  ) \
  .whenNotMatchedInsertAll() \
  .execute()